# リップシンク テスト (SadTalker on Colab)
静止画 `singer.png` + ボーカル `01Voice.wav` (9.3秒) から、口が音声に合う動画を作ります。

**進め方**: 上から順にセルを実行 (▶ボタン) してください。赤いエラーが出たら、その全文をマネージャーに送ってください。直したノートブックを返します。
まず メニュー「ランタイム」→「ランタイムのタイプを変更」→ GPU (T4) を選んで保存してから始めてください。

In [ ]:
# 1) GPU確認 (T4などと表示されればOK)
!nvidia-smi -L

In [ ]:
# 2) SadTalker を取得
%cd /content
!git clone -q https://github.com/OpenTalker/SadTalker.git
%cd /content/SadTalker
!ls

In [ ]:
# 3) 依存パッケージ (ColabのPython 3.13で組める版に差し替え済み)
!pip install -q numpy==1.26.4 face_alignment==1.3.5 imageio imageio-ffmpeg librosa==0.10.2.post1 resampy pydub scipy kornia==0.7.4 tqdm yacs pyyaml joblib scikit-image basicsr==1.4.2 facexlib==0.3.0 gfpgan av safetensors
# basicsr の新しいtorchvision互換パッチ (これが無いと生成時に落ちます)
import pathlib, basicsr
p = pathlib.Path(basicsr.__file__).parent / 'data' / 'degradations.py'
t = p.read_text()
p.write_text(t.replace('from torchvision.transforms.functional_tensor import rgb_to_grayscale',
                       'from torchvision.transforms.functional import rgb_to_grayscale'))
print("install done")

In [ ]:
# 4) モデル (チェックポイント) をダウンロード — 約2GB、数分かかります
!bash scripts/download_models.sh
!ls checkpoints

In [ ]:
# 5) 素材をアップロード: singer.png と 01Voice.wav の2つを選んでください
from google.colab import files
up = files.upload()
!cp -n singer.png 01Voice.wav /content/SadTalker/ 2>/dev/null; ls -la singer.png 01Voice.wav

In [ ]:
# 6) 生成 (9秒クリップ: T4で数分が目安)
%cd /content/SadTalker
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 python inference.py --driven_audio 01Voice.wav --source_image singer.png --result_dir ./results --still --preprocess full --enhancer gfpgan
!ls -la results

In [ ]:
# 7) 結果を表示してダウンロード
import glob
from IPython.display import Video, display
vids = sorted(glob.glob('results/*.mp4'))
print(vids)
display(Video(vids[-1], width=640, embed=True))
files.download(vids[-1])

## 終わったら
ダウンロードした mp4 をマネージャーに送ってください。フレームチェックして、必要なら連結や音声の仕上げをこちらでやります。
うまく行かない場合は、エラーが出たセルの赤い文字を全部コピーして送ってください。